# Evaluation — exact match per model and prompting strategy

Reports **Precision**, **Recall**, **F1** and **invalid** for each cell of the
model × prompting-strategy grid, under a single exact-match criterion.

All scoring lives in `clinical_notes_extraction.utils.llm.evaluation`, which mirrors
`schemas.py`; this notebook pins a run, checks it, and renders the tables.

**One slot** is one key of one annotation: `flag_is_medication_completed` plus the ten
attributes of each medication. A slot is a TP when the annotation holds a value and the
prediction reproduces it exactly, an FN when it does not, and an FP when the prediction
holds a value the annotation does not. Absent on both sides is a true negative and is not
counted, since rewarding agreement on emptiness would score every run highly for leaving
rarely used keys blank.

**Failed extractions count.** A note with no usable output has every annotated slot scored
as a false negative and creates no false positive, so recall and F1 absorb the failures
while precision does not. Read `invalid` next to them: high P with low R and high `invalid`
is a model that is accurate when it answers and often does not answer.

## 1. Setup

`RUN_ID` pins one extraction run; outputs from different runs are never mixed in a table.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
ROOT = next(p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / "src").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import json
import shutil

import numpy as np
import pandas as pd

from clinical_notes_extraction.utils.llm import evaluation as ev

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

SPLIT = "dev"                     # "dev" to select, "prod" for the final evaluation
RUN_ID = "20260822_194432"

DATA_DIR = NOTEBOOK_DIR / "data"
GROUND_TRUTH_DIR = DATA_DIR / "annotations" / "ground_truth" / SPLIT
RUN_DIR = DATA_DIR / "llm_extraction_results" / SPLIT / RUN_ID
OUTPUT_DIR = DATA_DIR / "llm_extraction_results" / SPLIT / "evaluation" / RUN_ID / "exact_match"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED, N_BOOTSTRAP = 20260814, 2000

if (RUN_DIR / "config.json").is_file():
    shutil.copy(RUN_DIR / "config.json", OUTPUT_DIR / "run_config.json")
    print(json.loads((RUN_DIR / "config.json").read_text(encoding="utf-8")))
else:
    print("WARNING: no config.json — the metrics are not tied to a decoding configuration")

## 2. Load and validate

Both sides are validated against `schemas.py` before anything is scored. This is the check
that matters most: `strict=True` and `extra="forbid"` mean a schema drift between the
annotations, the model output and the scored key list shows up here rather than as a
mysteriously low score. `ev.SCORED_KEYS` must match `MedicationAttributes` exactly.

In [ ]:
from clinical_notes_extraction.utils.llm.schemas import ExtractionOutput, GroundTruthNote, MedicationAttributes

gold_by_id = ev.load_ground_truth(GROUND_TRUTH_DIR)
records = ev.load_results(RUN_DIR)
NOTE_IDS = sorted(gold_by_id)

# the scored key list is the schema, or the evaluation is measuring the wrong thing
assert set(ev.ATTRIBUTES) == set(MedicationAttributes.model_fields), (
    set(ev.ATTRIBUTES) ^ set(MedicationAttributes.model_fields)
)

bad_gold = [n for n, note in gold_by_id.items()
            if not GroundTruthNote.model_validate(note, strict=True)]
bad_pred = []
for record in records:
    output = ev.usable_output(record)
    if output is None:
        continue
    try:
        ExtractionOutput.model_validate(output, strict=True)
    except Exception as error:
        bad_pred.append((record["path"], str(error)[:120]))

print(f"{len(NOTE_IDS)} annotated notes · {len(records)} result files")
print(f"{len(bad_gold)} invalid annotations · {len(bad_pred)} usable outputs failing the schema")
for path, error in bad_pred[:5]:
    print(" ", path, "->", error)

## 3. Coverage

Every cell must cover every annotated note. `missing` are notes with neither an output nor an
error file: they are scored as no output, but they normally mean the grid did not finish.
`duplicated` are notes present twice in a cell, which means the result tree is stale — the
scoring resolves them deterministically, but the tree should be regenerated.

In [ ]:
frame = pd.DataFrame(records)
frame["usable"] = frame.apply(lambda r: ev.usable_output(r.to_dict()) is not None, axis=1)
frame["annotated"] = frame.note_id.isin(NOTE_IDS)

coverage = (frame[frame.annotated].groupby(["model", "strategy"])
            .agg(usable=("usable", "sum"), unusable=("usable", lambda s: (~s).sum())))
coverage["missing"] = len(NOTE_IDS) - coverage.usable - coverage.unusable
coverage["duplicated"] = (frame[frame.annotated]
                          .groupby(["model", "strategy"]).note_id
                          .apply(lambda s: int(s.duplicated().sum())))
coverage["not_annotated"] = frame[~frame.annotated].groupby(["model", "strategy"]).size()
coverage.fillna(0).astype(int)

## 4. Score

One call. It returns three tidy frames — one row per slot, one per note, one per
disagreement — and every table below is a `groupby` on them.

In [ ]:
scored = ev.score(gold_by_id, records)
MODELS = sorted(scored.slots.model.unique())
print(f"{len(scored.slots):,} scored slots across {len(MODELS)} models")

## 5. Headline table

The table reported in the dissertation. `support` is the number of annotated slots, so no
score can be read without its denominator.

Precision and recall are not independent: a slot present on both sides but unequal counts as
one FP *and* one FN, so the two diverge only through omissions (recall) and inventions
(precision).

In [ ]:
headline = ev.headline_table(scored)
headline.round(3)

In [ ]:
for metric in ["P", "R", "F1", "invalid"]:
    print(f"\n=== {metric} ===")
    print(headline[metric].unstack("strategy").round(3))

## 6. Breakdown per key

Keys never annotated in this split are dropped: an all-zero row describes the split, not the
model. `Macro` is the unweighted mean over the keys shown; `Micro` pools their counts and
reproduces the headline F1. Precision shows `--` where it is undefined (nothing predicted for
that key), which is not the same as a measured zero.

In [ ]:
for model in MODELS:
    print(f"\n=== {model} ===")
    print(ev.per_key_table(scored, model=model).round(2).to_string())

## 7. Sanity checks on the alignment

Medications are matched before their attributes are compared, and the matcher has one free
parameter: the similarity below which two medications are not the same drug. If the ranking
holds across a range of thresholds, the reported figures are a property of the models; if it
does not, they are partly a property of the matcher. A mass of accepted pairs sitting just
above the threshold is the warning sign.

Detection is also reported here rather than in the headline: attribute scores are conditional
on it, and a missed medication is *already* inside the attribute micro-average as false
negatives, so demoting this table under-reports nothing.

In [ ]:
sensitivity = pd.DataFrame({
    f"F1@{t:.2f}": ev.headline_table(ev.score(gold_by_id, records, threshold=t))["F1"]
    for t in (0.10, 0.20, 0.30, 0.50)
})
sensitivity["range"] = sensitivity.max(axis=1) - sensitivity.min(axis=1)
display(sensitivity.round(3))

print("\nweakest accepted match per note:")
print(scored.notes.min_similarity.describe().round(3).to_string())
display(ev.medication_table(scored).round(3))

`medications_text` is reported here too. It is one span per note, so pooling it with the
attribute slots would let a whole section weigh the same as a single `route` — but the
template makes it a real extraction decision (find the heading, truncate before any
inpatient or discharge sub-section), so it is measured rather than dropped. `exact` is the
share of notes whose section span is reproduced exactly.

In [ ]:
ev.section_table(scored).round(3)

## 8. Does the winning difference survive resampling?

The grid selects a model–prompt combination for the held-out subset, so the selection needs
an uncertainty statement even though the reported table carries none. Notes are resampled
because they are the level at which the observations are independent, and both cells are
recomputed on the *same* resamples, which makes the comparison paired.

`separates` is the only column to act on: it is true when the 95% interval on the difference
excludes zero. Where it is false, the data do not distinguish the two cells — say so, and do
not report a ranking. `p_is_floor` marks rows where the p-value is an upper bound
(`p < 0.001` at 2 000 resamples) rather than a value.

In [ ]:
rng = np.random.default_rng(SEED)
BOOT = rng.integers(0, len(NOTE_IDS), size=(N_BOOTSTRAP, len(NOTE_IDS)))
matrices = ev.note_matrices(scored, NOTE_IDS)

intervals = pd.DataFrame({
    cell: np.percentile(ev.bootstrap_f1(matrix, BOOT), [2.5, 97.5])
    for cell, matrix in matrices.items()
}, index=["F1_lo", "F1_hi"]).T
intervals.index = pd.MultiIndex.from_tuples(intervals.index, names=["model", "strategy"])
intervals.join(headline["F1"]).sort_values("F1", ascending=False)[["F1", "F1_lo", "F1_hi"]].round(3)

In [ ]:
# strategies within a model, and models at a fixed strategy
# (the second is the controlled ablation the design exists to test)
cells = sorted(matrices)
comparisons = pd.DataFrame([
    ev.paired_bootstrap(matrices, a, b, BOOT)
    for i, a in enumerate(cells) for b in cells[i + 1:]
    if a[0] == b[0] or a[1] == b[1]
]).sort_values("delta_F1", ascending=False)
comparisons.round(3)

## 9. Material for the error analysis

One row per slot where the two sides differ, categorised by the first rule that applies:
`hallucinated` (annotation absent), `missed` (prediction absent), `casing`, `span_boundary`
(one value contained in the other), `different_value`. Only notes that produced valid output
are tabulated, since a failed note contributes nothing but `missed`.

The `casing` share answers the standard objection to exact match: if it is large, the
transcription convention rather than the model is driving the error.

In [ ]:
valid = scored.disagreements.query("valid")
assert not valid.empty, "no disagreement anywhere — verify the run before believing this"

display(valid.pivot_table(index=["model", "strategy"], columns="category",
                          values="note_id", aggfunc="count").fillna(0).astype(int))

display(valid.pivot_table(index="key", columns="category", values="note_id", aggfunc="count")
        .reindex(ev.SCORED_KEYS).dropna(how="all").fillna(0).astype(int))

## 10. Export

Every figure in the dissertation is generated here, so none is transcribed by hand.

In [ ]:
headline.to_csv(OUTPUT_DIR / "headline.csv")
scored.slots.to_csv(OUTPUT_DIR / "slots.csv", index=False)
ev.section_table(scored).to_csv(OUTPUT_DIR / 'section_span.csv')
scored.notes.to_csv(OUTPUT_DIR / "notes.csv", index=False)
scored.disagreements.to_csv(OUTPUT_DIR / "disagreements.csv", index=False)
comparisons.to_csv(OUTPUT_DIR / "paired_bootstrap.csv", index=False)
sensitivity.to_csv(OUTPUT_DIR / "threshold_sensitivity.csv")
coverage.to_csv(OUTPUT_DIR / "coverage.csv")

(OUTPUT_DIR / "settings.json").write_text(json.dumps({
    "run_id": RUN_ID, "split": SPLIT, "criterion": "exact_match",
    "seed": SEED, "n_bootstrap": N_BOOTSTRAP,
    "match_threshold": ev.MATCH_THRESHOLD,
    "scored_keys": list(ev.SCORED_KEYS),
    "order_insensitive_keys": sorted(ev.SET_KEYS),
    "array_order": "source order required by the template; arrays compared as ordered tuples",
    "failed_extractions": "scored as missing output (every annotated slot becomes FN)",
}, indent=2), encoding="utf-8")

ev.to_docx(
    {"Table 1: Exact-match Precision, Recall and F1 per model and prompting strategy. "
     "Failed extractions are counted as missing output; invalid is their proportion.":
        headline.round(3),
     **{f"Table {i + 2}: Exact-match results per key — {model}.":
        ev.per_key_table(scored, model=model).round(3)
        for i, model in enumerate(MODELS)},
     "Table 5: Exact match on the medications_text section span, reported separately.":
        ev.section_table(scored).round(3)},
    OUTPUT_DIR / f"exact_match_{RUN_ID}.docx",
    title=f"Exact-match evaluation — run {RUN_ID}",
)
print("written to", OUTPUT_DIR)